# Pensieve ML - Phase 2: Theme Discovery Demonstration

> **IMPORTANT NOTICE**: The journal entries analyzed in this notebook are simulated qualitative demonstration examples across various personal domains (academics, health, work, finances, etc.). They are **NOT** benchmark evaluation data.
> All discovered clusters reflect **unsupervised semantic text groupings**, not psychological diagnostics.

This notebook demonstrates the Sentence-BERT (`all-MiniLM-L6-v2`) + HDBSCAN theme-discovery pipeline for grouping user reflections into natural semantic topics.

In [ ]:
import sys
import os
import numpy as np

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ml.theme.embedding import TextEmbedder
from ml.theme.clustering import ThemeClusterer

print("Modules imported successfully.")

In [ ]:
# 1. Initialize Embedder & Generate Normalized Vectors
embedder = TextEmbedder()
sample_entries = [
    "Spent hours reviewing calculus problem sets for the midterm.",
    "Writing my thesis introduction chapter, advisor was encouraging.",
    "Pushed backend patch right before the team staging deployment freeze.",
    "Product sprint planning meeting with engineering leads.",
    "Ran 5k in the morning, feeling great and full of energy.",
    "Heavy lifting session at the gym focusing on deadlift form.",
    "Need to stick to my monthly budget and cut unnecessary expenses.",
    "Worried about rent and inflation, building an emergency fund."
]

embeddings = embedder.embed_texts(sample_entries)
print(f"Embeddings shape: {embeddings.shape} (L2 norm: {np.linalg.norm(embeddings[0]):.4f})")

In [ ]:
# 2. HDBSCAN Density Clustering
clusterer = ThemeClusterer(min_cluster_size=2, min_samples=1)
labels = clusterer.fit_predict(embeddings, texts=sample_entries)

for text, label in zip(sample_entries, labels):
    status = "OUTLIER" if label == -1 else f"Cluster {label}"
    print(f"[{status}] {text}")

In [ ]:
# 3. Inspect Discovered Clusters & Representative Central Entries
summary = clusterer.get_cluster_summary(top_n_examples=1)
for cid, info in summary["clusters"].items():
    print(f"\nCluster {cid} ({info['size']} entries) - Keywords: {info['simple_theme_descriptor']}")
    print(f"  Central representative: \"{info['representative_examples'][0]}\"")